# Startup App Success Analyzer

This notebook runs the project stage by stage and is suitable for an academic demonstration.


## 1. Architecture

Google Play metadata -> description embeddings -> semantic neighbors -> market features -> multiple classifiers -> selected model.


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print(ROOT)


d:\playstore_startup_success


## 2. Optional: collect live data

Run this once, then comment it out while experimenting to avoid repeated requests.


In [4]:
#%pip install -r requirements.txt
!python -m src.scrape_playstore --country us --lang en --delay 0.35


[search 1/48] fitness
[search 2/48] workout
[search 3/48] nutrition
[search 4/48] meditation
[search 5/48] mental wellness
[search 6/48] budget planner
[search 7/48] expense tracker
[search 8/48] personal finance
[search 9/48] investment
[search 10/48] education
[search 11/48] language learning
[search 12/48] study planner
[search 13/48] productivity
[search 14/48] task manager
[search 15/48] notes
[search 16/48] calendar
[search 17/48] habit tracker
[search 18/48] sleep tracker
[search 19/48] social network
[search 20/48] messaging
[search 21/48] photo editor
[search 22/48] video editor
[search 23/48] music
[search 24/48] podcast
[search 25/48] travel
[search 26/48] hotel booking
[search 27/48] food delivery
[search 28/48] recipe
[search 29/48] shopping
[search 30/48] fashion
[search 31/48] beauty
[search 32/48] health
[search 33/48] telemedicine
[search 34/48] weather
[search 35/48] news
[search 36/48] sports
[search 37/48] games
[search 38/48] puzzle
[search 39/48] strategy game
[se

## 3. Prepare data and create success target


In [2]:
from src.prepare_data import prepare
from src.config import RAW_APPS_CSV, PREPARED_CSV

df = prepare(RAW_APPS_CSV, PREPARED_CSV)
df[['title', 'genre', 'realInstalls', 'reviews', 'success_score', 'success']].head()


Prepared 1069 apps. Success rate: 24.8%


,title,genre,realInstalls,reviews,success_score,success
0,Learn AI & ML with Python,Education,434234,206.0,0.231818,0
1,"Character AI: Chat, Talk, Text",Entertainment,69583353,82715.0,0.768382,1
2,ChatOn - AI Chat Bot Assistant,Productivity,33701184,5038.0,0.743787,0
3,"KYRO: Storm, Linemen & VM",Productivity,3635,0.0,0.052367,0
4,Buddy.ai: Kids Learning Games,Education,64061986,2151.0,0.797727,1


## 4. Inspect target distribution


In [6]:
df['success'].value_counts(normalize=True).rename('share')


success
0    0.752105
1    0.247895
Name: share, dtype: float64

## 5. Generate semantic embeddings


In [3]:
import numpy as np
from src.embeddings import encode_descriptions
from src.config import EMBEDDINGS_NPY

embeddings = encode_descriptions(df['description'].tolist())
np.save(EMBEDDINGS_NPY, embeddings)
embeddings.shape


d:\playstore_startup_success\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 17/17 [03:41<00:00, 13.02s/it]


(1069, 384)

## 6. Build semantic competitor features


In [4]:
from src.features import build_training_features
from src.config import FEATURES_CSV

features = build_training_features(df, embeddings, top_k=30)
features.to_csv(FEATURES_CSV, index=False)
features.shape


(1069, 431)

## 7. Train and compare models


In [5]:
from src.train import train_models
from src.config import MODEL_BUNDLE, METRICS_JSON

bundle, metrics = train_models(FEATURES_CSV, MODEL_BUNDLE, METRICS_JSON)
metrics



Preparing leakage-safe cross-validation folds...
  Preparing CV fold 1/5...
  Preparing CV fold 2/5...
  Preparing CV fold 3/5...
  Preparing CV fold 4/5...
  Preparing CV fold 5/5...

Hyperparameter search: logistic_regression
  Config 1/3
  Config 2/3
  Config 3/3

Hyperparameter search: decision_tree
  Config 1/3
  Config 2/3
  Config 3/3

Hyperparameter search: random_forest
  Config 1/3
  Config 2/3
  Config 3/3

Hyperparameter search: extra_trees
  Config 1/3
  Config 2/3
  Config 3/3

Hyperparameter search: gradient_boosting
  Config 1/3
  Config 2/3
  Config 3/3

Hyperparameter search: hist_gradient_boosting
  Config 1/3
  Config 2/3
  Config 3/3

Hyperparameter search: adaboost
  Config 1/3
  Config 2/3
  Config 3/3

Hyperparameter search: svc
  Config 1/3


d:\playstore_startup_success\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
d:\playstore_startup_success\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
d:\playstore_startup_success\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
d:\playstore_startup_success\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 a

  Config 2/3


d:\playstore_startup_success\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
d:\playstore_startup_success\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
d:\playstore_startup_success\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
d:\playstore_startup_success\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 a

  Config 3/3


d:\playstore_startup_success\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
d:\playstore_startup_success\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
d:\playstore_startup_success\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
d:\playstore_startup_success\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 a


Hyperparameter search: knn
  Config 1/3
  Config 2/3
  Config 3/3

Hyperparameter search: gaussian_nb
  Config 1/3
  Config 2/3
  Config 3/3

Preparing development and comparison features...

Evaluating the best configuration of each model on the comparison set...
  Evaluating logistic_regression...
  Evaluating decision_tree...
  Evaluating random_forest...
  Evaluating extra_trees...
  Evaluating gradient_boosting...
  Evaluating hist_gradient_boosting...
  Evaluating adaboost...
  Evaluating svc...


d:\playstore_startup_success\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


  Evaluating knn...
  Evaluating gaussian_nb...

Selected model after comparison: extra_trees

Evaluating selected model on the untouched final test set...

Retraining selected model on all historical apps...

Selected model: extra_trees
{
  "accuracy": 0.7391304347826086,
  "precision": 0.4,
  "recall": 0.1,
  "f1": 0.16,
  "roc_auc": 0.7669421487603307,
  "pr_auc": 0.5019783411925106,
  "balanced_accuracy": 0.525206611570248,
  "specificity": 0.9504132231404959,
  "confusion_matrix": [
    [
      115,
      6
    ],
    [
      36,
      4
    ]
  ]
}

Evaluation report: D:\playstore_startup_success\artifacts\evaluation\model_evaluation_report.html


{'dataset': {'total_rows': 1069,
  'development_rows': 747,
  'comparison_rows': 161,
  'final_test_rows': 161,
  'positive_rate': 0.24789522918615528,
  'cv_folds': 5,
  'model_count': 10,
  'configuration_count': 30,
  'top_k': 30},
 'hyperparameter_search': [{'model': 'logistic_regression',
   'config_id': 1,
   'parameters': '{"C": 0.5, "class_weight": "balanced", "dual": false, "fit_intercept": true, "intercept_scaling": 1, "l1_ratio": 0.0, "max_iter": 3000, "n_jobs": null, "penalty": "deprecated", "random_state": 42, "solver": "lbfgs", "tol": 0.0001, "verbose": 0, "warm_start": false}',
   'cv_accuracy_mean': 0.6747024608501119,
   'cv_accuracy_std': 0.017598780065450608,
   'cv_precision_mean': 0.38095804195804195,
   'cv_precision_std': 0.02620588293972173,
   'cv_recall_mean': 0.5027027027027027,
   'cv_recall_std': 0.04712323182206132,
   'cv_f1_mean': 0.43312285156288466,
   'cv_f1_std': 0.03326878018387208,
   'cv_roc_auc_mean': 0.6910009567089214,
   'cv_roc_auc_std': 0.02

## 8. Test a startup idea


In [6]:
from src.predict import StartupSuccessPredictor, StartupInput

predictor = StartupSuccessPredictor()
result = predictor.predict(StartupInput(
    description='An AI-powered fitness coach that creates personalized workouts, tracks nutrition, adapts routines from progress, and helps users build sustainable exercise habits.',
    summary='AI personal workout and nutrition coach',
    genre='Health & Fitness',
    content_rating='Everyone',
    free=True,
    offers_iap=True,
    contains_ads=False,
), top_k=30)

print('Probability:', f"{result['probability']:.1%}")
result['neighbors'].head(10)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1170.30it/s]


Probability: 17.5%


,title,appId,genre,score,realInstalls,reviews,success,similarity,url
431,AI Fitness: Personal Trainer,com.heaox.wellness.aifitness,Health & Fitness,0.000000,242,0.0,0,0.813152,https://play.google.com/store/apps/details?id=...
983,Simple・AI Weight Loss & Health,life.simple,Health & Fitness,4.502094,7440335,13713.0,0,0.719601,https://play.google.com/store/apps/details?id=...
357,Fitness AI Pro,com.gauravgangwar.fitnessaipro,Health & Fitness,0.000000,16,0.0,0,0.715198,https://play.google.com/store/apps/details?id=...
778,Gymertia - AI Gym Trainer,com.teknikforce.gym_trainer,Health & Fitness,0.000000,16656,0.0,0,0.714542,https://play.google.com/store/apps/details?id=...
334,FitSense AI: AI Fitness Coach,com.fitsense.ai,Health & Fitness,0.000000,209,0.0,0,0.714074,https://play.google.com/store/apps/details?id=...
106,AI Fitness,com.altryscryptoworks.aifitness,Health & Fitness,0.000000,62,0.0,0,0.704262,https://play.google.com/store/apps/details?id=...
1031,F/AI: AI Gym & Fitness Trainer,pro.fitgpt.fai,Health & Fitness,4.222222,110989,8.0,0,0.702815,https://play.google.com/store/apps/details?id=...
566,Planfit - Gym Workout Planner,com.mih.planfit,Health & Fitness,4.627451,1347746,194.0,0,0.702094,https://play.google.com/store/apps/details?id=...
328,Fitbod: Workout & Gym Planner,com.fitbod.fitbod,Health & Fitness,4.457668,4682741,3941.0,0,0.694794,https://play.google.com/store/apps/details?id=...
819,UltraFit360: AI Fitness Coach,com.ultrafit360,Health & Fitness,0.000000,5687,0.0,0,0.673602,https://play.google.com/store/apps/details?id=...


In [7]:
from src.predict import StartupSuccessPredictor, StartupInput

predictor = StartupSuccessPredictor()
result = predictor.predict(StartupInput(
    description='a game where you can build your own houses and design them with furniture, decorations, and more. Players can create their dream homes and share them with friends.players can play with other players in the same world as well as visit their friends houses and see how they have designed their homes. The game also features a marketplace where players can buy and sell furniture and decorations to customize their homes even further.',
    summary='build homes',
    genre='game',
    content_rating='Everyone',
    free=True,
    offers_iap=True,
    contains_ads=False,
), top_k=30)

print('Probability:', f"{result['probability']:.1%}")
result['neighbors'].head(10)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3963.75it/s]


Probability: 44.8%


,title,appId,genre,score,realInstalls,reviews,success,similarity,url
652,Homescapes,com.playrix.homescapes,Casual,4.728310,521493061,370839.0,1,0.553232,https://play.google.com/store/apps/details?id=...
793,Toca Boca World,com.tocaboca.tocalifeworld,Educational,4.348820,454948051,217914.0,1,0.483857,https://play.google.com/store/apps/details?id=...
651,Gardenscapes,com.playrix.gardenscapes,Casual,4.753134,605442054,354499.0,1,0.479910,https://play.google.com/store/apps/details?id=...
633,Avatar World ®,com.pazugames.avatarworld,Role Playing,4.727198,263322519,74350.0,1,0.461095,https://play.google.com/store/apps/details?id=...
346,Fashion Empire - Dressup Sim,com.frenzoo.FashionEmpireBoutiqueGirlGame,Role Playing,4.365517,26179801,20454.0,0,0.419020,https://play.google.com/store/apps/details?id=...
155,Makeup Games For Kids: Salon,com.bimiboo.kids.games.beauty.salon,Educational,4.627451,531754,11.0,0,0.418305,https://play.google.com/store/apps/details?id=...
573,Minecraft Education,com.mojang.minecraftedu,Education,4.136234,60738523,10443.0,1,0.410109,https://play.google.com/store/apps/details?id=...
370,"Fashion Show: Makeup, Dress Up",com.gofiveglobal.fashion.dress.up,Simulation,4.552110,269361745,8173.0,1,0.404710,https://play.google.com/store/apps/details?id=...
653,Township,com.playrix.township,Casual,4.687940,520862415,374146.0,1,0.404627,https://play.google.com/store/apps/details?id=...
698,Coloring Games: Color & Paint,com.rvappstudios.kids.coloring.book.color.pain...,Educational,4.403385,181038670,1041.0,0,0.396418,https://play.google.com/store/apps/details?id=...
